In [ ]:
import sys
import os
from pathlib import Path

# 1. (혹시 옮겼다면) 원래 디렉터리로 원복하거나 현재 파일 위치 기준으로 설정
# os.chdir("원래폴더경로") 

# 2. narajangteo... 파일이 있는 'tools' 폴더의 절대 경로 찾기
# 현재 실행 위치 기준으로 tools 폴더 경로 지정 (프로젝트 구조에 맞게 수정)
BASE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path('.').resolve()
TOOLS_DIR = BASE_DIR / "backend_logic2" / "nodes" / "tools"

# 3. sys.path에 tools 폴더 경로 등록 (기존 코드 수정 없이 import 가능해짐)
if str(TOOLS_DIR) not in sys.path:
    sys.path.append(str(TOOLS_DIR))

# 4. 이제 기존 코드 변경 없이 그대로 import
from backend_logic2.nodes.tools.narajangteo_search_based_tool import get_corp_basic_info

In [1]:
import pandas as pd
import io
import re

pd.set_option("display.max_columns", None)

In [16]:
with open("조달업체_등록내역.csv", "rb") as f:
    raw = f.read()

text = raw.decode("utf-16-le", errors="replace")
df = pd.read_csv(io.StringIO(text))

print(f"원본: {len(df)}행, {len(df.columns)}컬럼")
df.head()

C:\Users\박동관\AppData\Local\Temp\ipykernel_7428\3038592203.py:5: DtypeWarning: Columns (0: 사업자등록번호) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(text))


원본: 658796행, 14컬럼


,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,여성기업인증여부,장애인기업인증여부,사회적기업인증여부
0,제 일 인 테 리 어,5150419799,대구광역시 동구,본사,대한민국,중소기업,기타자유업종,N,NaN,NaN,20260605,N,N,N
1,주경수산,8279901276,전북특별자치도 김제시,본사,대한민국,중소기업,NaN,N,5.012161e+09,신선한조개,20250326,NaN,NaN,NaN
2,Bulneth Holland BV,F000000303,국외소재 기타지역,본사,네덜란드,중소기업,NaN,N,NaN,NaN,20100524,N,N,N
3,Dalsem Horticultural projects B.V,F000000302,국외소재 기타지역,본사,네덜란드,중소기업,NaN,N,NaN,NaN,20100524,N,N,N
4,Societe Des Anciens Etablissement L Geismar,F000000307,국외소재 기타지역,본사,프랑스,중소기업,NaN,N,NaN,NaN,20100609,N,N,N


In [17]:
df["업체국가"].value_counts()

업체국가
대한민국         658090
미국              190
일본               64
중국               59
독일               58
영국               35
프랑스              31
싱가포르             27
캐나다              25
스위스              17
이탈리아             15
홍콩               15
네덜란드             14
몽골               13
스페인              11
오스트레일리아          10
오스트리아             9
벨기에               9
스웨덴               9
노르웨이              9
러시아               8
덴마크               7
아랍에미리트            6
체코                6
이스라엘              6
아일랜드              4
핀란드               4
타이완               4
폴란드               4
말레이시아             3
남극                3
태국                2
슬로베니아             2
전세계               2
뉴질랜드              2
인도                2
우크라이나             2
나이지리아             2
버뮤다               2
아프가니스탄            1
인도네시아             1
슬로바키아             1
아르헨티나             1
남아프리카 공화국         1
캄보디아              1
팔라우               1
에콰도르              1
칠레                1
스리랑카              1
미얀마            

In [18]:
df = df[df["업체국가"] == "대한민국"].reset_index(drop=True)
print(f"필터 후: {len(df)}행")

필터 후: 658090행


In [19]:
df['기업구분'].value_counts()

기업구분
중소기업        636732
비영리법인등기타     15424
중견기업          4609
대기업           1325
Name: count, dtype: int64

In [20]:
## 비영리법인등기타 제외
df = df[df['기업구분'] != '비영리법인등기타'].reset_index(drop=True)
print(f"필터 후: {len(df)}행")

필터 후: 642666행


In [26]:
df

,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,여성기업인증여부,장애인기업인증여부,사회적기업인증여부
0,제 일 인 테 리 어,5150419799,대구광역시 동구,본사,대한민국,중소기업,기타자유업종,N,NaN,NaN,20260605,N,N,N
1,주경수산,8279901276,전북특별자치도 김제시,본사,대한민국,중소기업,NaN,N,5.012161e+09,신선한조개,20250326,NaN,NaN,NaN
2,경원농자재,2780801307,강원특별자치도 원주시,본사,대한민국,중소기업,NaN,N,7.212120e+09,온실설치공사,20260519,Y,NaN,NaN
3,광일중기,3120866954,충청남도 천안시 동남구,본사,대한민국,중소기업,건설기계대여업 연명신고사업자,N,NaN,NaN,20250813,N,N,N
4,그린테크,6800102500,강원특별자치도 원주시,본사,대한민국,중소기업,기타자유업종,N,4.810180e+09,정수기,20250108,N,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
642661,힛더마크,4093402568,경상북도 구미시,본사,대한민국,중소기업,NaN,N,5.215150e+09,일회용접시또는식기,20240603,NaN,NaN,NaN
642662,힛더마크,4093402568,경상북도 구미시,본사,대한민국,중소기업,NaN,N,5.310231e+09,유아용기저귀,20240603,NaN,NaN,NaN
642663,힛더스페이스,7381001990,경기도 안양시 동안구,본사,대한민국,중소기업,기타자유업종,N,NaN,NaN,20220516,NaN,NaN,NaN
642664,힛텍,1304288817,경기도 부천시 원미구,본사,대한민국,중소기업,NaN,Y,4.010182e+09,투입히터,20240517,N,N,N


In [29]:
df['대표업종'].value_counts()

대표업종
기타자유업종                  68139
전기공사업                   21244
소프트웨어사업자(컴퓨터관련서비스사업)    13572
건축사사무소                  13261
학술.연구용역                 11920
                        ...  
폐수배출시설(기타섬유제품제조시설)          1
기술사사무소(섬유)                  1
박제품판매업                      1
승강기제조업(휠체어리프트 조립제조)         1
대부업                         1
Name: count, Length: 1590, dtype: int64

In [30]:
before = len(df)
df = df.dropna(subset=['대표업종', '대표세부품명번호', '대표세부품명']).reset_index(drop=True)
print(f"필터 전: {before}행 -> 필터 후: {len(df)}행 ({before - len(df)}행 제거)")

필터 전: 642666행 -> 필터 후: 155733행 (486933행 제거)


In [31]:
df

,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,여성기업인증여부,장애인기업인증여부,사회적기업인증여부
0,그린테크,6800102500,강원특별자치도 원주시,본사,대한민국,중소기업,기타자유업종,N,4.810180e+09,정수기,20250108,N,N,N
1,서천해초김 주식회사,4118192223,충청남도 서천군,본사,대한민국,중소기업,식품제조·가공업,Y,5.012180e+09,조미김,20250307,NaN,NaN,NaN
2,선경수산,1979601946,경기도 양평군,본사,대한민국,중소기업,수산종자생산업(육상수조식),N,5.012154e+09,신선한생선,20260304,Y,NaN,NaN
3,스튜디오안온,8661802174,인천광역시 연수구,본사,대한민국,중소기업,기타자유업(행사대행업),N,7.215410e+09,전시부스설치및디자인서비스,20250410,Y,NaN,NaN
4,에이엔씨헬스케어주식회사,5628702507,경기도 김포시,본사,대한민국,중소기업,동물용의료기기판매업,N,4.212156e+09,동물용초음파영상진단장치,20260429,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155728,힘찬푸드,4871102986,대전광역시 대덕구,본사,대한민국,중소기업,식품판매업(집단급식소식품판매업),N,5.046700e+09,배추김치,20241206,NaN,NaN,NaN
155729,힘택스,5990402490,서울특별시 강남구,본사,대한민국,중소기업,세무사사무소,N,8.411160e+09,회계서비스,20231211,Y,NaN,NaN
155730,힙 프로덕션,2290402233,경기도 하남시,본사,대한민국,중소기업,방송영상독립제작자,N,8.213160e+09,동영상제작서비스,20240403,Y,NaN,NaN
155731,힙유한책임회사,4438101302,경기도 수원시 팔달구,본사,대한민국,중소기업,소프트웨어사업자(컴퓨터관련서비스사업),N,8.111200e+09,빅데이터분석서비스,20200304,N,N,N


In [32]:
df['나라장터등록일자'] = pd.to_datetime(df['나라장터등록일자'].astype(str), format='%Y%m%d', errors='coerce')

before = len(df)
df = df[df['나라장터등록일자'] >= '2020-01-01'].reset_index(drop=True)
print(f"필터 전: {before}행 -> 필터 후: {len(df)}행 ({before - len(df)}행 제거)")

필터 전: 155733행 -> 필터 후: 62507행 (93226행 제거)


In [33]:
df

,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,여성기업인증여부,장애인기업인증여부,사회적기업인증여부
0,그린테크,6800102500,강원특별자치도 원주시,본사,대한민국,중소기업,기타자유업종,N,4.810180e+09,정수기,2025-01-08,N,N,N
1,서천해초김 주식회사,4118192223,충청남도 서천군,본사,대한민국,중소기업,식품제조·가공업,Y,5.012180e+09,조미김,2025-03-07,NaN,NaN,NaN
2,선경수산,1979601946,경기도 양평군,본사,대한민국,중소기업,수산종자생산업(육상수조식),N,5.012154e+09,신선한생선,2026-03-04,Y,NaN,NaN
3,스튜디오안온,8661802174,인천광역시 연수구,본사,대한민국,중소기업,기타자유업(행사대행업),N,7.215410e+09,전시부스설치및디자인서비스,2025-04-10,Y,NaN,NaN
4,에이엔씨헬스케어주식회사,5628702507,경기도 김포시,본사,대한민국,중소기업,동물용의료기기판매업,N,4.212156e+09,동물용초음파영상진단장치,2026-04-29,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62502,힘찬푸드,4871102986,대전광역시 대덕구,본사,대한민국,중소기업,식품판매업(집단급식소식품판매업),N,5.046700e+09,배추김치,2024-12-06,NaN,NaN,NaN
62503,힘택스,5990402490,서울특별시 강남구,본사,대한민국,중소기업,세무사사무소,N,8.411160e+09,회계서비스,2023-12-11,Y,NaN,NaN
62504,힙 프로덕션,2290402233,경기도 하남시,본사,대한민국,중소기업,방송영상독립제작자,N,8.213160e+09,동영상제작서비스,2024-04-03,Y,NaN,NaN
62505,힙유한책임회사,4438101302,경기도 수원시 팔달구,본사,대한민국,중소기업,소프트웨어사업자(컴퓨터관련서비스사업),N,8.111200e+09,빅데이터분석서비스,2020-03-04,N,N,N


In [34]:
##사업자 번호 없는것만 걸러내기 
before = len(df)
df = df.dropna(subset=['사업자등록번호']).reset_index(drop=True)
print(f"필터 전: {before}행 -> 필터 후: {len(df)}행 ({before - len(df)}행 제거)")

필터 전: 62507행 -> 필터 후: 62507행 (0행 제거)


In [37]:
import time
from urllib.parse import unquote
import requests

API_URL = "https://api.odcloud.kr/api/nts-businessman/v1/status"
service_key = unquote(os.environ["DATA_GO_KR_SERVICE_KEY"])

def check_business_status(bizno_list, max_retries=3):
    clean_list = [str(b).replace("-", "").strip() for b in bizno_list]
    for attempt in range(max_retries):
        try:
            res = requests.post(
                API_URL,
                params={"serviceKey": service_key},
                headers={"Content-Type": "application/json", "Accept": "application/json"},
                json={"b_no": clean_list},
                timeout=30,
            )
            res.raise_for_status()
            return res.json().get("data", [])
        except requests.exceptions.Timeout:
            print(f"    타임아웃, 재시도 {attempt + 1}/{max_retries}")
            time.sleep(1)
        except Exception as e:
            print(f"    오류: {e}, 재시도 {attempt + 1}/{max_retries}")
            time.sleep(1)
    print(f"    최종 실패, 이 배치({len(clean_list)}건) 건너뜀")
    return []

unique_biznos = df["사업자등록번호"].dropna().astype(str).unique().tolist()
status_map = {}

for i in range(0, len(unique_biznos), 100):
    batch = unique_biznos[i:i + 100]
    results = check_business_status(batch)
    for r in results:
        status_map[r["b_no"]] = r.get("b_stt_cd")
    print(f"진행: {i + len(batch)}/{len(unique_biznos)}")
    time.sleep(0.2)

df["사업자상태코드"] = df["사업자등록번호"].astype(str).map(status_map)

failed_count = df["사업자상태코드"].isna().sum()
if failed_count > 0:
    print(f"조회 실패(건너뛴 배치 포함)로 상태 못 얻은 행: {failed_count}건 - 필터에서 제외 안 되고 남아있음")

before = len(df)
df = df[df["사업자상태코드"] != "03"].reset_index(drop=True)
print(f"필터 전: {before}행 -> 필터 후: {len(df)}행")

진행: 100/59963
진행: 200/59963
진행: 300/59963
진행: 400/59963
진행: 500/59963
진행: 600/59963
진행: 700/59963
진행: 800/59963
진행: 900/59963
진행: 1000/59963
진행: 1100/59963
진행: 1200/59963
진행: 1300/59963
진행: 1400/59963
진행: 1500/59963
진행: 1600/59963
진행: 1700/59963
진행: 1800/59963
진행: 1900/59963
진행: 2000/59963
진행: 2100/59963
진행: 2200/59963
진행: 2300/59963
진행: 2400/59963
진행: 2500/59963
진행: 2600/59963
진행: 2700/59963
진행: 2800/59963
진행: 2900/59963
진행: 3000/59963
진행: 3100/59963
진행: 3200/59963
진행: 3300/59963
진행: 3400/59963
진행: 3500/59963
진행: 3600/59963
진행: 3700/59963
진행: 3800/59963
진행: 3900/59963
진행: 4000/59963
진행: 4100/59963
진행: 4200/59963
진행: 4300/59963
진행: 4400/59963
진행: 4500/59963
진행: 4600/59963
진행: 4700/59963
진행: 4800/59963
진행: 4900/59963
진행: 5000/59963
진행: 5100/59963
진행: 5200/59963
진행: 5300/59963
진행: 5400/59963
진행: 5500/59963
진행: 5600/59963
진행: 5700/59963
진행: 5800/59963
진행: 5900/59963
진행: 6000/59963
진행: 6100/59963
진행: 6200/59963
진행: 6300/59963
진행: 6400/59963
진행: 6500/59963
진행: 6600/59963
진행: 6700/59963
진행: 

In [39]:
df = df.drop(columns=["여성기업인증여부", "장애인기업인증여부", "사회적기업인증여부"])
print(f"남은 컬럼: {list(df.columns)}")

남은 컬럼: ['업체명', '사업자등록번호', '업체소재시군구', '본사지사구분', '업체국가', '기업구분', '대표업종', '제조업체여부', '대표세부품명번호', '대표세부품명', '나라장터등록일자', '사업자상태코드']


In [40]:
df

,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,사업자상태코드
0,그린테크,6800102500,강원특별자치도 원주시,본사,대한민국,중소기업,기타자유업종,N,4.810180e+09,정수기,2025-01-08,01
1,서천해초김 주식회사,4118192223,충청남도 서천군,본사,대한민국,중소기업,식품제조·가공업,Y,5.012180e+09,조미김,2025-03-07,01
2,선경수산,1979601946,경기도 양평군,본사,대한민국,중소기업,수산종자생산업(육상수조식),N,5.012154e+09,신선한생선,2026-03-04,01
3,스튜디오안온,8661802174,인천광역시 연수구,본사,대한민국,중소기업,기타자유업(행사대행업),N,7.215410e+09,전시부스설치및디자인서비스,2025-04-10,01
4,에이엔씨헬스케어주식회사,5628702507,경기도 김포시,본사,대한민국,중소기업,동물용의료기기판매업,N,4.212156e+09,동물용초음파영상진단장치,2026-04-29,01
...,...,...,...,...,...,...,...,...,...,...,...,...
58176,힘찬푸드,3099952791,인천광역시 남동구,본사,대한민국,중소기업,식육포장처리업,N,2.711270e+09,전동드릴,2025-03-12,01
58177,힘택스,5990402490,서울특별시 강남구,본사,대한민국,중소기업,세무사사무소,N,8.411160e+09,회계서비스,2023-12-11,01
58178,힙 프로덕션,2290402233,경기도 하남시,본사,대한민국,중소기업,방송영상독립제작자,N,8.213160e+09,동영상제작서비스,2024-04-03,01
58179,힙유한책임회사,4438101302,경기도 수원시 팔달구,본사,대한민국,중소기업,소프트웨어사업자(컴퓨터관련서비스사업),N,8.111200e+09,빅데이터분석서비스,2020-03-04,01


In [ ]:
#test
def search_by_item(keyword, top_n=10):
    matches = df[df["대표세부품명"].str.contains(keyword, na=False, case=False)]
    print(f"'{keyword}' 검색결과: {len(matches)}건 (상위 {min(top_n, len(matches))}건 표시)")
    return matches[["업체명", "사업자등록번호", "대표세부품명", "대표업종", "제조업체여부"]].head(top_n)

# 테스트
search_by_item("")

'잉크' 검색결과: 12건 (상위 10건 표시)


,업체명,사업자등록번호,대표세부품명,대표업종,제조업체여부
10646,드림OA,5142321027,잉크젯프린터,기타자유업종,N
15742,비엘솔루션주식회사,6238801829,잉크젯프린터,기타자유업종,N
17086,서울정인사,3957600223,스탬프잉크,기타자유업종,N
19298,신도중앙OA,2101398168,잉크젯프린터,기타자유업종,N
26801,유한회사 정도컴퍼니,6598703421,잉크카트리지,시약판매업,N
29449,제이월드,2152413893,잉크젯프린터,기타자유업종,N
29665,제일전산,2591100122,잉크젯프린터,소프트웨어사업자(컴퓨터관련서비스사업),N
41997,주식회사 웨이드,3328802975,잉크젯프린터,소프트웨어사업자(컴퓨터관련서비스사업),Y
42428,주식회사 유진기업,5218102275,잉크카트리지,의료기기판매업,N
43624,주식회사 정연오에이,7448801387,잉크젯프린터,기타자유업종,N


In [86]:
def search_local_registry_ranked(item_name, top_n=10):
    """
    CSV 로컬검색, AI 호출 없이 길이순 정렬 (짧을수록 범용적 = 관련도 높음).
    """
    all_categories = df["대표세부품명"].dropna().unique().tolist()
    candidates = [c for c in all_categories if item_name in c]

    if not candidates:
        print(f"'{item_name}' 매칭되는 카테고리 없음")
        return []

    # 길이순 정렬 (짧은 것 = 범용적인 것 우선)
    candidates_sorted = sorted(candidates, key=len)

    print(f"'{item_name}' 관련도순 후보:")
    for c in candidates_sorted:
        print(f"  {c}  ({len(c)}자)")

    best_category = candidates_sorted[0]  # 가장 짧은(=가장 범용적인) 것 채택
    print(f"\n-> 채택: {best_category}")

    matches = df[df["대표세부품명"] == best_category].head(top_n)

    results = []
    for _, row in matches.iterrows():
        results.append({
            "name": row["업체명"],
            "bizno": row["사업자등록번호"],
            "category": row["대표세부품명"],
            "industry": row["대표업종"],
        })
    return results


# 테스트
results = search_local_registry_ranked("선풍기")
for r in results:
    print(r)

'선풍기' 관련도순 후보:
  선풍기  (3자)
  디젤기관차용냉각선풍기  (11자)

-> 채택: 선풍기
{'name': '라온정책연구소', 'bizno': '2640902721', 'category': '선풍기', 'industry': '건물(시설)관리용역'}
{'name': '라온정책연구소', 'bizno': '2640902721', 'category': '선풍기', 'industry': '기타자유업(행사대행업)'}
{'name': '라온정책연구소', 'bizno': '2640902721', 'category': '선풍기', 'industry': '기타자유업종'}
{'name': '라온정책연구소', 'bizno': '2640902721', 'category': '선풍기', 'industry': '학술.연구용역'}
{'name': '스페이스나다', 'bizno': '2050436381', 'category': '선풍기', 'industry': '학술.연구용역'}
{'name': '신영산업', 'bizno': '7900602006', 'category': '선풍기', 'industry': '기타자유업종'}
{'name': '어울림연구소', 'bizno': '2443701304', 'category': '선풍기', 'industry': '건물(시설)관리용역'}
{'name': '에이치알엠', 'bizno': '4691200731', 'category': '선풍기', 'industry': '기타자유업종'}
{'name': '엠에이치상사', 'bizno': '3151663136', 'category': '선풍기', 'industry': '시약판매업'}
{'name': '엠와이지', 'bizno': '6271500546', 'category': '선풍기', 'industry': '기타자유업종'}


In [116]:
import os
print(os.getcwd())

c:\Users\박동관\Desktop\SKN31-FINAL-3Team


In [113]:
from backend_logic2.nodes.tools.narajangteo_search_based_tool import get_corp_basic_info
from backend_logic2.nodes.tools.naver_contact_enrichment import _fetch_page_text, _extract_contacts_batch

def enrich_from_bizno(company_name, bizno):
    basic_info = get_corp_basic_info(bizno)
    if not basic_info:
        return {"name": company_name, "phone": None, "email": None, "homepage": None}

    phone = basic_info.get("telNo")  # 여기서 이미 확보, AI 필요없음
    homepage = basic_info.get("hmpgAdrs")

    email = None
    if homepage:
        page_text = _fetch_page_text(homepage)  # 네이버검색 없이 바로 그 페이지로
        result = _extract_contacts_batch([{"name": company_name, "page_text": page_text}])
        email = result.get(company_name, {}).get("email")

    return {"name": company_name, "phone": phone, "email": email, "homepage": homepage}

# 테스트
results = search_local_registry_ranked("목재")
for r in results[:3]:
    enriched = enrich_from_bizno(r["name"], r["bizno"])
    print(enriched)

'목재' 관련도순 후보:
  목재덱  (3자)
  목재칩  (3자)
  합성목재  (4자)
  목재판재  (4자)
  목재펠릿  (4자)
  집성목재  (4자)
  목재파쇄차  (5자)
  목재파쇄기  (5자)
  목재플라스틱혼합바닥재  (11자)
  목재 및 건축자재 중개분야 전기전자복합  (21자)
  주방용 및 음식점용 목재 가구 제조분야 기계장비복합  (28자)

-> 채택: 목재덱
{'name': '(주)일호산업', 'phone': '070-4252-1515', 'email': None, 'homepage': ''}
{'name': '(주)준디자인조경', 'phone': '070-4251-0318', 'email': None, 'homepage': ''}
{'name': '(주)푸르다산림조경지점', 'phone': '032-577-4970', 'email': None, 'homepage': ''}


In [117]:
df

,업체명,사업자등록번호,업체소재시군구,본사지사구분,업체국가,기업구분,대표업종,제조업체여부,대표세부품명번호,대표세부품명,나라장터등록일자,사업자상태코드
0,그린테크,6800102500,강원특별자치도 원주시,본사,대한민국,중소기업,기타자유업종,N,4.810180e+09,정수기,2025-01-08,01
1,서천해초김 주식회사,4118192223,충청남도 서천군,본사,대한민국,중소기업,식품제조·가공업,Y,5.012180e+09,조미김,2025-03-07,01
2,선경수산,1979601946,경기도 양평군,본사,대한민국,중소기업,수산종자생산업(육상수조식),N,5.012154e+09,신선한생선,2026-03-04,01
3,스튜디오안온,8661802174,인천광역시 연수구,본사,대한민국,중소기업,기타자유업(행사대행업),N,7.215410e+09,전시부스설치및디자인서비스,2025-04-10,01
4,에이엔씨헬스케어주식회사,5628702507,경기도 김포시,본사,대한민국,중소기업,동물용의료기기판매업,N,4.212156e+09,동물용초음파영상진단장치,2026-04-29,01
...,...,...,...,...,...,...,...,...,...,...,...,...
58176,힘찬푸드,3099952791,인천광역시 남동구,본사,대한민국,중소기업,식육포장처리업,N,2.711270e+09,전동드릴,2025-03-12,01
58177,힘택스,5990402490,서울특별시 강남구,본사,대한민국,중소기업,세무사사무소,N,8.411160e+09,회계서비스,2023-12-11,01
58178,힙 프로덕션,2290402233,경기도 하남시,본사,대한민국,중소기업,방송영상독립제작자,N,8.213160e+09,동영상제작서비스,2024-04-03,01
58179,힙유한책임회사,4438101302,경기도 수원시 팔달구,본사,대한민국,중소기업,소프트웨어사업자(컴퓨터관련서비스사업),N,8.111200e+09,빅데이터분석서비스,2020-03-04,01


In [118]:
output_path = "exported_data.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")